In [ ]:
# healthy +unhealthy
# Logistic Regression baseline (0 vs 1) on your tiles
# 0: healthy + moderately_healthy + moderately_unhealthy
# 1: unhealthy + severely_unhealthy
# Uses simple pixel features (tiny resized image flattened)
# ============================================================

from pathlib import Path
from PIL import Image
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ----------------------------
# Paths + class mapping
# ----------------------------
TILES_ROOT = Path("/home/ehr/Desktop/anupama/plant/class_tiles")

NEG_CLASSES = {"healthy", "moderately_healthy", "moderately_unhealthy"}   # -> 0
POS_CLASSES = {"unhealthy", "severely_unhealthy"}                         # -> 1

BIN_NAMES = ["healthy_or_modhealthy_or_modunhealthy", "unhealthy_or_severe"]

IMG_FEAT_SIZE = 48  # feature image size (small keeps RAM ok). try 32/48/64

def is_image(p: Path) -> bool:
    return p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def load_split(split: str):
    X_list, y_list = [], []

    split_dir = TILES_ROOT / split
    assert split_dir.exists(), split_dir

    for cls_dir in sorted(split_dir.iterdir()):
        if not cls_dir.is_dir():
            continue
        cls_name = cls_dir.name

        # only use known classes
        if cls_name not in (NEG_CLASSES | POS_CLASSES):
            continue

        y = 0 if cls_name in NEG_CLASSES else 1
        img_dir = cls_dir / "images"
        if not img_dir.exists():
            continue

        for p in sorted(img_dir.rglob("*")):
            if not (p.is_file() and is_image(p)):
                continue

            im = Image.open(p).convert("RGB").resize((IMG_FEAT_SIZE, IMG_FEAT_SIZE))
            arr = np.asarray(im, dtype=np.float32) / 255.0  # (H,W,3) in [0,1]
            feat = arr.reshape(-1)  # flatten
            X_list.append(feat)
            y_list.append(y)

    X = np.stack(X_list, axis=0)
    y = np.array(y_list, dtype=np.int64)
    return X, y

# ----------------------------
# Load data
# ----------------------------
X_train, y_train = load_split("train")
X_val,   y_val   = load_split("val")

print("Train shape:", X_train.shape, "Counts:", np.bincount(y_train))
print("Val   shape:", X_val.shape,   "Counts:", np.bincount(y_val))

# ----------------------------
# Model: scaled Logistic Regression
# class_weight='balanced' helps imbalance
# ----------------------------
clf = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("lr", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs",
        n_jobs=-1
    ))
])

clf.fit(X_train, y_train)

# ----------------------------
# Evaluate
# ----------------------------
pred = clf.predict(X_val)
acc = accuracy_score(y_val, pred)
macro = f1_score(y_val, pred, average="macro")

print("Val accuracy:", acc)
print("Val macro F1:", macro)
print("\nConfusion matrix:\n", confusion_matrix(y_val, pred))
print("\nReport:\n", classification_report(y_val, pred, target_names=BIN_NAMES, digits=4, zero_division=0))


Train shape: (15139, 6912) Counts: [14259   880]
Val   shape: (2153, 6912) Counts: [2075   78]


/home/ehr/Desktop/anupama/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Val accuracy: 0.9122155132373433
Val macro F1: 0.6173106782694401

Confusion matrix:
 [[1927  148]
 [  41   37]]

Report:
                                        precision    recall  f1-score   support

healthy_or_modhealthy_or_modunhealthy     0.9792    0.9287    0.9533      2075
                  unhealthy_or_severe     0.2000    0.4744    0.2814        78

                             accuracy                         0.9122      2153
                            macro avg     0.5896    0.7015    0.6173      2153
                         weighted avg     0.9509    0.9122    0.9289      2153



healthy vs rest

In [4]:
# ============================================================
# BLOCK A: Logistic Regression (Healthy vs Rest)
# 0: healthy
# 1: moderately_healthy + moderately_unhealthy + unhealthy + severely_unhealthy
# ============================================================

from pathlib import Path
from PIL import Image
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ----------------------------
# Paths + class mapping
# ----------------------------
TILES_ROOT = Path("/home/ehr/Desktop/anupama/plant/class_tiles")

NEG_CLASSES = {"healthy"}   # -> 0
POS_CLASSES = {"moderately_healthy", "moderately_unhealthy", "unhealthy", "severely_unhealthy"}  # -> 1

BIN_NAMES = ["healthy", "rest"]

IMG_FEAT_SIZE = 48

def is_image(p: Path) -> bool:
    return p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def load_split(split: str):
    X_list, y_list = [], []

    split_dir = TILES_ROOT / split
    if not split_dir.exists():
        raise FileNotFoundError(split_dir)

    for cls_dir in sorted(split_dir.iterdir()):
        if not cls_dir.is_dir():
            continue

        cls_name = cls_dir.name
        if cls_name not in (NEG_CLASSES | POS_CLASSES):
            continue

        y = 0 if cls_name in NEG_CLASSES else 1

        for p in sorted(cls_dir.rglob("*")):
            if not (p.is_file() and is_image(p)):
                continue

            with Image.open(p) as im:
                im = im.convert("RGB").resize((IMG_FEAT_SIZE, IMG_FEAT_SIZE))
                arr = np.asarray(im, dtype=np.float32) / 255.0
                feat = arr.reshape(-1)

            X_list.append(feat)
            y_list.append(y)

    if len(X_list) == 0:
        raise RuntimeError(f"No images found in {split_dir}")

    return np.stack(X_list), np.array(y_list)

# ----------------------------
# Load data
# ----------------------------
X_train, y_train = load_split("train")
X_val, y_val = load_split("val")

print("Train:", X_train.shape, np.bincount(y_train))
print("Val:", X_val.shape, np.bincount(y_val))

# ----------------------------
# Model
# ----------------------------
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs"))
])

clf.fit(X_train, y_train)

# ----------------------------
# Evaluate
# ----------------------------
pred = clf.predict(X_val)

print("Accuracy:", accuracy_score(y_val, pred))
print("Macro F1:", f1_score(y_val, pred, average="macro", zero_division=0))
print("\nConfusion Matrix:\n", confusion_matrix(y_val, pred))
print("\nReport:\n", classification_report(y_val, pred, target_names=BIN_NAMES, digits=4, zero_division=0))

Train: (30278, 6912) [11600 18678]
Val: (4306, 6912) [1934 2372]
Accuracy: 0.7113330236878774
Macro F1: 0.7113261577800136

Confusion Matrix:
 [[1521  413]
 [ 830 1542]]

Report:
               precision    recall  f1-score   support

     healthy     0.6470    0.7865    0.7099      1934
        rest     0.7887    0.6501    0.7127      2372

    accuracy                         0.7113      4306
   macro avg     0.7179    0.7183    0.7113      4306
weighted avg     0.7251    0.7113    0.7115      4306



/home/ehr/Desktop/anupama/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


mod healthy vs rest

In [7]:
# ============================================================
# BLOCK B: Logistic Regression (Moderately Healthy vs Rest)
# 0: moderately_healthy
# 1: healthy + moderately_unhealthy + unhealthy + severely_unhealthy
# ============================================================

from pathlib import Path
from PIL import Image
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ----------------------------
# Paths + class mapping
# ----------------------------
TILES_ROOT = Path("/home/ehr/Desktop/anupama/plant/class_tiles")

NEG_CLASSES = {"moderately_healthy"}   # -> 0
POS_CLASSES = {"healthy", "moderately_unhealthy", "unhealthy", "severely_unhealthy"}  # -> 1

BIN_NAMES = ["moderately_healthy", "rest"]

IMG_FEAT_SIZE = 48

def is_image(p: Path) -> bool:
    return p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def load_split(split: str):
    X_list, y_list = [], []

    split_dir = TILES_ROOT / split
    if not split_dir.exists():
        raise FileNotFoundError(split_dir)

    for cls_dir in sorted(split_dir.iterdir()):
        if not cls_dir.is_dir():
            continue

        cls_name = cls_dir.name
        if cls_name not in (NEG_CLASSES | POS_CLASSES):
            continue

        y = 0 if cls_name in NEG_CLASSES else 1

        for p in sorted(cls_dir.rglob("*")):
            if not (p.is_file() and is_image(p)):
                continue

            with Image.open(p) as im:
                im = im.convert("RGB").resize((IMG_FEAT_SIZE, IMG_FEAT_SIZE))
                arr = np.asarray(im, dtype=np.float32) / 255.0
                feat = arr.reshape(-1)

            X_list.append(feat)
            y_list.append(y)

    if len(X_list) == 0:
        raise RuntimeError(f"No images found in {split_dir}")

    return np.stack(X_list), np.array(y_list)

# ----------------------------
# Load data
# ----------------------------
X_train, y_train = load_split("train")
X_val, y_val = load_split("val")

print("Train:", X_train.shape, np.bincount(y_train))
print("Val:", X_val.shape, np.bincount(y_val))

# ----------------------------
# Model
# ----------------------------
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs"))
])

clf.fit(X_train, y_train)

# ----------------------------
# Evaluate
# ----------------------------
pred = clf.predict(X_val)

print("Accuracy:", accuracy_score(y_val, pred))
print("Macro F1:", f1_score(y_val, pred, average="macro", zero_division=0))
print("\nConfusion Matrix:\n", confusion_matrix(y_val, pred))
print("\nReport:\n", classification_report(y_val, pred, target_names=BIN_NAMES, digits=4, zero_division=0))

Train: (30278, 6912) [10782 19496]
Val: (4306, 6912) [1372 2934]
Accuracy: 0.570599163957269
Macro F1: 0.5025118428301647

Confusion Matrix:
 [[ 432  940]
 [ 909 2025]]

Report:
                     precision    recall  f1-score   support

moderately_healthy     0.3221    0.3149    0.3185      1372
              rest     0.6830    0.6902    0.6866      2934

          accuracy                         0.5706      4306
         macro avg     0.5026    0.5025    0.5025      4306
      weighted avg     0.5680    0.5706    0.5693      4306



/home/ehr/Desktop/anupama/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


mod unhealthy vs rest

In [6]:
# ============================================================
# BLOCK C: Logistic Regression (Moderately Unhealthy vs All Other Classes)
# 0: moderately_unhealthy
# 1: healthy + moderately_healthy + unhealthy + severely_unhealthy
# ============================================================

from pathlib import Path
from PIL import Image
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ----------------------------
# Paths + class mapping
# ----------------------------
TILES_ROOT = Path("/home/ehr/Desktop/anupama/plant/class_tiles")

NEG_CLASSES = {"moderately_unhealthy"}   # -> 0
POS_CLASSES = {"healthy", "moderately_healthy", "unhealthy", "severely_unhealthy"}  # -> 1

BIN_NAMES = ["moderately_unhealthy", "all_other_classes"]

IMG_FEAT_SIZE = 48

def is_image(p: Path) -> bool:
    return p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def load_split(split: str):
    X_list, y_list = [], []

    split_dir = TILES_ROOT / split
    if not split_dir.exists():
        raise FileNotFoundError(split_dir)

    for cls_dir in sorted(split_dir.iterdir()):
        if not cls_dir.is_dir():
            continue

        cls_name = cls_dir.name
        if cls_name not in (NEG_CLASSES | POS_CLASSES):
            continue

        y = 0 if cls_name in NEG_CLASSES else 1

        for p in sorted(cls_dir.rglob("*")):
            if not (p.is_file() and is_image(p)):
                continue

            with Image.open(p) as im:
                im = im.convert("RGB").resize((IMG_FEAT_SIZE, IMG_FEAT_SIZE))
                arr = np.asarray(im, dtype=np.float32) / 255.0
                feat = arr.reshape(-1)

            X_list.append(feat)
            y_list.append(y)

    if len(X_list) == 0:
        raise RuntimeError(f"No images found in {split_dir}")

    return np.stack(X_list), np.array(y_list)

# ----------------------------
# Load data
# ----------------------------
X_train, y_train = load_split("train")
X_val, y_val = load_split("val")

print("Train:", X_train.shape, np.bincount(y_train))
print("Val:", X_val.shape, np.bincount(y_val))

# ----------------------------
# Model
# ----------------------------
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs"
    ))
])

clf.fit(X_train, y_train)

# ----------------------------
# Evaluate
# ----------------------------
pred = clf.predict(X_val)

print("Accuracy:", accuracy_score(y_val, pred))
print("Macro F1:", f1_score(y_val, pred, average="macro", zero_division=0))
print("\nConfusion Matrix:\n", confusion_matrix(y_val, pred))
print("\nReport:\n", classification_report(y_val, pred, target_names=BIN_NAMES, digits=4, zero_division=0))

Train: (30278, 6912) [ 6136 24142]
Val: (4306, 6912) [ 844 3462]
Accuracy: 0.710868555503948
Macro F1: 0.6147298907964054

Confusion Matrix:
 [[ 455  389]
 [ 856 2606]]

Report:
                       precision    recall  f1-score   support

moderately_unhealthy     0.3471    0.5391    0.4223       844
   all_other_classes     0.8701    0.7527    0.8072      3462

            accuracy                         0.7109      4306
           macro avg     0.6086    0.6459    0.6147      4306
        weighted avg     0.7676    0.7109    0.7317      4306



/home/ehr/Desktop/anupama/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


unhealthy vs rest

In [9]:
# ============================================================
# BLOCK D: Logistic Regression (Unhealthy vs Rest)
# 0: unhealthy
# 1: healthy + moderately_healthy + moderately_unhealthy + severely_unhealthy
# ============================================================

from pathlib import Path
from PIL import Image
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ----------------------------
# Paths + class mapping
# ----------------------------
TILES_ROOT = Path("/home/ehr/Desktop/anupama/plant/class_tiles")

NEG_CLASSES = {"unhealthy"}   # -> 0
POS_CLASSES = {"healthy", "moderately_healthy", "moderately_unhealthy", "severely_unhealthy"}  # -> 1

BIN_NAMES = ["unhealthy", "rest"]

IMG_FEAT_SIZE = 48

def is_image(p: Path) -> bool:
    return p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def load_split(split: str):
    X_list, y_list = [], []

    split_dir = TILES_ROOT / split
    if not split_dir.exists():
        raise FileNotFoundError(split_dir)

    for cls_dir in sorted(split_dir.iterdir()):
        if not cls_dir.is_dir():
            continue

        cls_name = cls_dir.name
        if cls_name not in (NEG_CLASSES | POS_CLASSES):
            continue

        y = 0 if cls_name in NEG_CLASSES else 1

        for p in sorted(cls_dir.rglob("*")):
            if not (p.is_file() and is_image(p)):
                continue

            with Image.open(p) as im:
                im = im.convert("RGB").resize((IMG_FEAT_SIZE, IMG_FEAT_SIZE))
                arr = np.asarray(im, dtype=np.float32) / 255.0
                feat = arr.reshape(-1)

            X_list.append(feat)
            y_list.append(y)

    if len(X_list) == 0:
        raise RuntimeError(f"No images found in {split_dir}")

    return np.stack(X_list), np.array(y_list)

# ----------------------------
# Load data
# ----------------------------
X_train, y_train = load_split("train")
X_val, y_val = load_split("val")

print("Train:", X_train.shape, np.bincount(y_train))
print("Val:", X_val.shape, np.bincount(y_val))

# ----------------------------
# Model
# ----------------------------
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs"
    ))
])

clf.fit(X_train, y_train)

# ----------------------------
# Evaluate
# ----------------------------
pred = clf.predict(X_val)

print("Accuracy:", accuracy_score(y_val, pred))
print("Macro F1:", f1_score(y_val, pred, average="macro", zero_division=0))
print("\nConfusion Matrix:\n", confusion_matrix(y_val, pred))
print("\nReport:\n", classification_report(y_val, pred, target_names=BIN_NAMES, digits=4, zero_division=0))

Train: (30278, 6912) [ 1136 29142]
Val: (4306, 6912) [ 116 4190]
Accuracy: 0.8708778448676265
Macro F1: 0.5710691274027901

Confusion Matrix:
 [[  75   41]
 [ 515 3675]]

Report:
               precision    recall  f1-score   support

   unhealthy     0.1271    0.6466    0.2125       116
        rest     0.9890    0.8771    0.9297      4190

    accuracy                         0.8709      4306
   macro avg     0.5580    0.7618    0.5711      4306
weighted avg     0.9657    0.8709    0.9104      4306



/home/ehr/Desktop/anupama/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


severly unhealthy vs rest

In [8]:
# ============================================================
# BLOCK E: Logistic Regression (Severely Unhealthy vs All Other Classes)
# 0: severely_unhealthy
# 1: healthy + moderately_healthy + moderately_unhealthy + unhealthy
# ============================================================

from pathlib import Path
from PIL import Image
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ----------------------------
# Paths + class mapping
# ----------------------------
TILES_ROOT = Path("/home/ehr/Desktop/anupama/plant/class_tiles")

NEG_CLASSES = {"severely_unhealthy"}   # -> 0
POS_CLASSES = {"healthy", "moderately_healthy", "moderately_unhealthy", "unhealthy"}  # -> 1

BIN_NAMES = ["severely_unhealthy", "all_other_classes"]

IMG_FEAT_SIZE = 48

def is_image(p: Path) -> bool:
    return p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def load_split(split: str):
    X_list, y_list = [], []

    split_dir = TILES_ROOT / split
    if not split_dir.exists():
        raise FileNotFoundError(split_dir)

    for cls_dir in sorted(split_dir.iterdir()):
        if not cls_dir.is_dir():
            continue

        cls_name = cls_dir.name
        if cls_name not in (NEG_CLASSES | POS_CLASSES):
            continue

        y = 0 if cls_name in NEG_CLASSES else 1

        for p in sorted(cls_dir.rglob("*")):
            if not (p.is_file() and is_image(p)):
                continue

            with Image.open(p) as im:
                im = im.convert("RGB").resize((IMG_FEAT_SIZE, IMG_FEAT_SIZE))
                arr = np.asarray(im, dtype=np.float32) / 255.0
                feat = arr.reshape(-1)

            X_list.append(feat)
            y_list.append(y)

    if len(X_list) == 0:
        raise RuntimeError(f"No images found in {split_dir}")

    return np.stack(X_list), np.array(y_list)

# ----------------------------
# Load data
# ----------------------------
X_train, y_train = load_split("train")
X_val, y_val = load_split("val")

print("Train:", X_train.shape, np.bincount(y_train))
print("Val:", X_val.shape, np.bincount(y_val))

# ----------------------------
# Model
# ----------------------------
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs"
    ))
])

clf.fit(X_train, y_train)

# ----------------------------
# Evaluate
# ----------------------------
pred = clf.predict(X_val)

print("Accuracy:", accuracy_score(y_val, pred))
print("Macro F1:", f1_score(y_val, pred, average="macro", zero_division=0))
print("\nConfusion Matrix:\n", confusion_matrix(y_val, pred))
print("\nReport:\n", classification_report(y_val, pred, target_names=BIN_NAMES, digits=4, zero_division=0))

Train: (30278, 6912) [  624 29654]
Val: (4306, 6912) [  40 4266]
Accuracy: 0.9816535067347887
Macro F1: 0.7111654710222144

Confusion Matrix:
 [[  30   10]
 [  69 4197]]

Report:
                     precision    recall  f1-score   support

severely_unhealthy     0.3030    0.7500    0.4317        40
 all_other_classes     0.9976    0.9838    0.9907      4266

          accuracy                         0.9817      4306
         macro avg     0.6503    0.8669    0.7112      4306
      weighted avg     0.9912    0.9817    0.9855      4306



In [5]:
import pickle
from pathlib import Path
from PIL import Image
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

TILES_ROOT = Path("/home/ehr/Desktop/anupama/plant/class_tiles")
OUT_ROOT = Path("/home/ehr/Desktop/anupama/plant/training_runs")
IMG_FEAT_SIZE = 48

STAGES = [
    ("stage1_healthy_vs_rest", {"healthy"}, {"moderately_healthy", "moderately_unhealthy", "unhealthy", "severely_unhealthy"}, ["healthy", "rest"]),
    ("stage2_modhealthy_vs_rest", {"moderately_healthy"}, {"moderately_unhealthy", "unhealthy", "severely_unhealthy"}, ["moderately_healthy", "rest"]),
    ("stage3_modunhealthy_vs_rest", {"moderately_unhealthy"}, {"unhealthy", "severely_unhealthy"}, ["moderately_unhealthy", "rest"]),
    ("stage4_unhealthy_vs_rest", {"unhealthy"}, {"severely_unhealthy"}, ["unhealthy", "rest"]),
]

def is_image(p: Path) -> bool:
    return p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def load_split(split: str, neg_classes, pos_classes):
    X_list, y_list = [], []
    split_dir = TILES_ROOT / split

    for cls_dir in sorted(split_dir.iterdir()):
        if not cls_dir.is_dir():
            continue

        cls_name = cls_dir.name
        if cls_name not in (neg_classes | pos_classes):
            continue

        y = 0 if cls_name in neg_classes else 1

        for p in sorted(cls_dir.rglob("*")):
            if p.is_file() and is_image(p):
                with Image.open(p) as im:
                    im = im.convert("RGB").resize((IMG_FEAT_SIZE, IMG_FEAT_SIZE))
                    arr = np.asarray(im, dtype=np.float32) / 255.0
                    X_list.append(arr.reshape(-1))
                    y_list.append(y)

    if not X_list:
        raise RuntimeError(f"No images found in {split_dir}")

    return np.stack(X_list), np.array(y_list)

for stage_name, neg_classes, pos_classes, bin_names in STAGES:
    print(f"\n===== {stage_name} =====")

    X_train, y_train = load_split("train", neg_classes, pos_classes)
    X_val, y_val = load_split("val", neg_classes, pos_classes)

    print("Train:", X_train.shape, np.bincount(y_train))
    print("Val:", X_val.shape, np.bincount(y_val))

    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            solver="saga",
            random_state=42
        ))
    ])

    clf.fit(X_train, y_train)
    pred = clf.predict(X_val)

    print("Accuracy:", accuracy_score(y_val, pred))
    print("Macro F1:", f1_score(y_val, pred, average="macro", zero_division=0))
    print("\nConfusion Matrix:\n", confusion_matrix(y_val, pred))
    print("\nReport:\n", classification_report(y_val, pred, target_names=bin_names, digits=4, zero_division=0))

    out_dir = OUT_ROOT / stage_name
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "logistic_model.pkl", "wb") as f:
        pickle.dump(clf, f)


===== stage1_healthy_vs_rest =====
Train: (30278, 6912) [11600 18678]
Val: (4306, 6912) [1934 2372]
Accuracy: 0.7268927078495123
Macro F1: 0.7268912349009273

Confusion Matrix:
 [[1560  374]
 [ 802 1570]]

Report:
               precision    recall  f1-score   support

     healthy     0.6605    0.8066    0.7263      1934
        rest     0.8076    0.6619    0.7275      2372

    accuracy                         0.7269      4306
   macro avg     0.7340    0.7343    0.7269      4306
weighted avg     0.7415    0.7269    0.7270      4306


===== stage2_modhealthy_vs_rest =====
Train: (18678, 6912) [10782  7896]
Val: (2372, 6912) [1372 1000]
Accuracy: 0.6829679595278246
Macro F1: 0.6719869776323328

Confusion Matrix:
 [[1027  345]
 [ 407  593]]

Report:
                     precision    recall  f1-score   support

moderately_healthy     0.7162    0.7485    0.7320      1372
              rest     0.6322    0.5930    0.6120      1000

          accuracy                         0.6830      2

In [7]:
# ============================================================
# Hierarchical Inference: Logistic Regression Version
#
# Uses 5 saved logistic regression models:
#   stage1: healthy vs rest
#   stage2: moderately_healthy vs rest
#   stage3: moderately_unhealthy vs rest
#   stage4: unhealthy vs rest
#   stage5: severely_unhealthy vs rest
#
# Output: final 5-class prediction + stage probabilities
# ============================================================

import numpy as np
import pickle
from PIL import Image
from pathlib import Path

# Paths to saved logistic regression models
OUT_ROOT = Path("/home/ehr/Desktop/anupama/plant/training_runs")
STAGE1_LR = OUT_ROOT / "stage1_healthy_vs_rest" / "logistic_model.pkl"
STAGE2_LR = OUT_ROOT / "stage2_modhealthy_vs_rest" / "logistic_model.pkl"
STAGE3_LR = OUT_ROOT / "stage3_modunhealthy_vs_rest" / "logistic_model.pkl"
STAGE4_LR = OUT_ROOT / "stage4_unhealthy_vs_rest" / "logistic_model.pkl"
STAGE5_LR = OUT_ROOT / "stage5_severelyunhealthy_vs_rest" / "logistic_model.pkl"

# Final 5-class label order
FINAL_CLASS_NAMES = [
    "healthy",
    "moderately_healthy",
    "moderately_unhealthy",
    "unhealthy",
    "severely_unhealthy",
]

# Image feature extraction size
IMG_FEAT_SIZE = 48

def load_logistic_model(model_path):
    if not model_path.exists():
        raise FileNotFoundError(f"Logistic regression model not found: {model_path}")
    with open(model_path, "rb") as f:
        return pickle.load(f)

# Load all logistic regression models
stage1_model = load_logistic_model(STAGE1_LR)
stage2_model = load_logistic_model(STAGE2_LR)
stage3_model = load_logistic_model(STAGE3_LR)
stage4_model = load_logistic_model(STAGE4_LR)
stage5_model = load_logistic_model(STAGE5_LR)

def preprocess_image(img_path):
    img_path = Path(img_path)
    if not img_path.exists():
        raise FileNotFoundError(f"Image not found: {img_path}")

    with Image.open(img_path) as im:
        im = im.convert("RGB").resize((IMG_FEAT_SIZE, IMG_FEAT_SIZE))
        arr = np.asarray(im, dtype=np.float32) / 255.0
        feature_vector = arr.reshape(-1)

    return feature_vector

def predict_stage(model, X):
    probs = model.predict_proba([X])[0]
    pred = int(np.argmax(probs))
    return pred, probs

def predict_hierarchical_logistic(image_path):
    X = preprocess_image(image_path)

    # Stage 1: healthy vs rest
    s1_pred, s1_probs = predict_stage(stage1_model, X)
    if s1_pred == 0:
        return {
            "image_path": str(image_path),
            "final_id": 0,
            "final_name": FINAL_CLASS_NAMES[0],
            "stages": {
                "stage1": {"pred": s1_pred, "probs": s1_probs.tolist()}
            }
        }

    # Stage 2: moderately_healthy vs rest
    s2_pred, s2_probs = predict_stage(stage2_model, X)
    if s2_pred == 0:
        return {
            "image_path": str(image_path),
            "final_id": 1,
            "final_name": FINAL_CLASS_NAMES[1],
            "stages": {
                "stage1": {"pred": s1_pred, "probs": s1_probs.tolist()},
                "stage2": {"pred": s2_pred, "probs": s2_probs.tolist()},
            }
        }

    # Stage 3: moderately_unhealthy vs rest
    s3_pred, s3_probs = predict_stage(stage3_model, X)
    if s3_pred == 0:
        return {
            "image_path": str(image_path),
            "final_id": 2,
            "final_name": FINAL_CLASS_NAMES[2],
            "stages": {
                "stage1": {"pred": s1_pred, "probs": s1_probs.tolist()},
                "stage2": {"pred": s2_pred, "probs": s2_probs.tolist()},
                "stage3": {"pred": s3_pred, "probs": s3_probs.tolist()},
            }
        }

    # Stage 4: unhealthy vs rest
    s4_pred, s4_probs = predict_stage(stage4_model, X)
    if s4_pred == 0:
        return {
            "image_path": str(image_path),
            "final_id": 3,
            "final_name": FINAL_CLASS_NAMES[3],
            "stages": {
                "stage1": {"pred": s1_pred, "probs": s1_probs.tolist()},
                "stage2": {"pred": s2_pred, "probs": s2_probs.tolist()},
                "stage3": {"pred": s3_pred, "probs": s3_probs.tolist()},
                "stage4": {"pred": s4_pred, "probs": s4_probs.tolist()},
            }
        }

    # Stage 5: severely_unhealthy vs rest
    s5_pred, s5_probs = predict_stage(stage5_model, X)

    return {
        "image_path": str(image_path),
        "final_id": 4 if s5_pred == 0 else 3,
        "final_name": FINAL_CLASS_NAMES[4 if s5_pred == 0 else 3],
        "stages": {
            "stage1": {"pred": s1_pred, "probs": s1_probs.tolist()},
            "stage2": {"pred": s2_pred, "probs": s2_probs.tolist()},
            "stage3": {"pred": s3_pred, "probs": s3_probs.tolist()},
            "stage4": {"pred": s4_pred, "probs": s4_probs.tolist()},
            "stage5": {"pred": s5_pred, "probs": s5_probs.tolist()},
        }
    }

# ============================================================
# Example usage
# Replace this with a real image path from your dataset
# ============================================================

example_image_path = "/home/ehr/Desktop/anupama/plant/class_tiles/val/healthy/your_image.jpg"

result = predict_hierarchical_logistic(example_image_path)
print("Final Prediction:", result["final_id"], result["final_name"])
print("Stage Details:", result["stages"])

FileNotFoundError: Image not found: /home/ehr/Desktop/anupama/plant/class_tiles/val/healthy/your_image.jpg

In [1]:
# ============================================================
# Hierarchical Inference: Logistic Regression Version
# ============================================================

import numpy as np
import pickle
from PIL import Image
from pathlib import Path

OUT_ROOT = Path("/home/ehr/Desktop/anupama/plant/training_runs")
STAGE1_LR = OUT_ROOT / "stage1_healthy_vs_rest" / "logistic_model.pkl"
STAGE2_LR = OUT_ROOT / "stage2_modhealthy_vs_rest" / "logistic_model.pkl"
STAGE3_LR = OUT_ROOT / "stage3_modunhealthy_vs_rest" / "logistic_model.pkl"
STAGE4_LR = OUT_ROOT / "stage4_unhealthy_vs_rest" / "logistic_model.pkl"

FINAL_CLASS_NAMES = [
    "healthy",
    "moderately_healthy",
    "moderately_unhealthy",
    "unhealthy",
    "severely_unhealthy",
]

IMG_FEAT_SIZE = 48
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def is_image_file(p: Path) -> bool:
    return p.suffix.lower() in IMAGE_EXTS

def load_logistic_model(model_path):
    if not model_path.exists():
        raise FileNotFoundError(f"Logistic regression model not found: {model_path}")
    with open(model_path, "rb") as f:
        return pickle.load(f)

stage1_model = load_logistic_model(STAGE1_LR)
stage2_model = load_logistic_model(STAGE2_LR)
stage3_model = load_logistic_model(STAGE3_LR)
stage4_model = load_logistic_model(STAGE4_LR)

def preprocess_image(img_path):
    img_path = Path(img_path)
    if not img_path.exists():
        raise FileNotFoundError(f"Image not found: {img_path}")

    with Image.open(img_path) as im:
        im = im.convert("RGB").resize((IMG_FEAT_SIZE, IMG_FEAT_SIZE))
        arr = np.asarray(im, dtype=np.float32) / 255.0
        feature_vector = arr.reshape(-1)

    return feature_vector

def predict_stage(model, X):
    probs = model.predict_proba([X])[0]
    pred = int(np.argmax(probs))
    p0 = float(probs[0])
    p1 = float(probs[1])
    return pred, p0, p1, probs

def predict_hierarchical_logistic(image_path):
    X = preprocess_image(image_path)
    stages = {}

    # Stage 1: healthy vs rest
    s1_pred, s1_p0, s1_p1, s1_probs = predict_stage(stage1_model, X)
    stages["stage1"] = {
        "pred": s1_pred,
        "p0": s1_p0,
        "p1": s1_p1,
        "probs": s1_probs.tolist(),
        "class_0_name": "healthy",
        "class_1_name": "rest",
    }
    if s1_pred == 0:
        return {
            "image_path": str(image_path),
            "final_id": 0,
            "final_name": FINAL_CLASS_NAMES[0],
            "stages": stages,
        }

    # Stage 2: moderately_healthy vs rest
    s2_pred, s2_p0, s2_p1, s2_probs = predict_stage(stage2_model, X)
    stages["stage2"] = {
        "pred": s2_pred,
        "p0": s2_p0,
        "p1": s2_p1,
        "probs": s2_probs.tolist(),
        "class_0_name": "moderately_healthy",
        "class_1_name": "rest",
    }
    if s2_pred == 0:
        return {
            "image_path": str(image_path),
            "final_id": 1,
            "final_name": FINAL_CLASS_NAMES[1],
            "stages": stages,
        }

    # Stage 3: moderately_unhealthy vs rest
    s3_pred, s3_p0, s3_p1, s3_probs = predict_stage(stage3_model, X)
    stages["stage3"] = {
        "pred": s3_pred,
        "p0": s3_p0,
        "p1": s3_p1,
        "probs": s3_probs.tolist(),
        "class_0_name": "moderately_unhealthy",
        "class_1_name": "rest",
    }
    if s3_pred == 0:
        return {
            "image_path": str(image_path),
            "final_id": 2,
            "final_name": FINAL_CLASS_NAMES[2],
            "stages": stages,
        }

    # Stage 4: unhealthy vs severely_unhealthy
    s4_pred, s4_p0, s4_p1, s4_probs = predict_stage(stage4_model, X)
    stages["stage4"] = {
        "pred": s4_pred,
        "p0": s4_p0,
        "p1": s4_p1,
        "probs": s4_probs.tolist(),
        "class_0_name": "unhealthy",
        "class_1_name": "severely_unhealthy",
    }

    final_id = 3 if s4_pred == 0 else 4

    return {
        "image_path": str(image_path),
        "final_id": final_id,
        "final_name": FINAL_CLASS_NAMES[final_id],
        "stages": stages,
    }

# ============================================================
# Example usage: auto pick one real image
# ============================================================

example_dir = Path("/home/ehr/Desktop/anupama/plant/class_tiles/val/healthy")
example_candidates = [p for p in sorted(example_dir.rglob("*")) if p.is_file() and is_image_file(p)]

if not example_candidates:
    raise RuntimeError(f"No image files found in {example_dir}")

example_image_path = example_candidates[0]

result = predict_hierarchical_logistic(example_image_path)
print("Image:", example_image_path)
print("Final Prediction:", result["final_id"], result["final_name"])
print("Stage Details:")
for stage_name, stage_info in result["stages"].items():
    print(
        f"  {stage_name}: "
        f"{stage_info['class_0_name']}={stage_info['p0']:.4f}, "
        f"{stage_info['class_1_name']}={stage_info['p1']:.4f}, "
        f"pred={stage_info['pred']}"
    )

Image: /home/ehr/Desktop/anupama/plant/class_tiles/val/healthy/images/DJI_0004_JPG.rf.cce0b194885278359ac873c978785315_c1_blob10_x4956y1586w192h192.jpg
Final Prediction: 0 healthy
Stage Details:
  stage1: healthy=0.6242, rest=0.3758, pred=0
